# P1-D: exact first Schur correction assembly

This notebook reuses the executed P1-C construction, assembles the source-backed seven-factor correction exactly, and then demonstrates the independent fail-closed algebra and symbol gates.

In [1]:
from pathlib import Path
import sys
import sympy as sp

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))
from symbolic_operator_calculus import OperatorAtom  # noqa: E402
p1c_notebook = repo_root / 'notebooks' / 'bilateral_diagonal_regularizer.ipynb'
get_ipython().run_line_magic('run', str(p1c_notebook))
p1c_globals = globals()
result = p1c_globals['result']
gamma = p1c_globals['gamma']
space = p1c_globals['space']
assumptions = p1c_globals['assumptions']
x = p1c_globals['x']
lam = p1c_globals['lam']
compact_ideal = p1c_globals['compact_ideal']
multiplier = p1c_globals['multiplier']
print('P1-C center:', result.status.value)

A11: A11
T1_minus: LinearCombination(terms=(Term(coefficient=1, product=Product(factors=(OperatorAtom(name='U1_inverse', kind='operator'), OperatorAtom(name='P_plus', kind='operator')))), Term(coefficient=1, product=Product(factors=(OperatorAtom(name='P_minus', kind='operator'),)))))
Z1: (-I*c_1 + gamma*x)/(gamma**kappa*(-I*c_1 + x))
Z1_inverse: gamma**kappa*(-I*c_1 + x)/(-I*c_1 + gamma*x)
H1 (P1-C alias of N1): H1
R1: R1 / TWO_SIDED_REGULARIZER
link: Product(factors=(OperatorAtom(name='Z1_inverse', kind='operator'), OperatorAtom(name='A11', kind='operator'), OperatorAtom(name='T1_minus', kind='operator'))) = Product(factors=(OperatorAtom(name='H1', kind='operator'),))
candidate: Product(factors=(OperatorAtom(name='T1_minus', kind='operator'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='Z1_inverse', kind='operator')))
left product: Product(factors=(OperatorAtom(name='A11', kind='operator'), OperatorAtom(name='T1_minus', kind='operator'), OperatorAtom(name='R1

In [2]:
from symbolic_operator_calculus import (  # noqa: E402
    Branch, BranchDifferenceOperator, BranchedDiagonalOperator,
    CoefficientMultiplicationOperator, LocalizedOperator, OffDiagonalBlockModel,
    SchurExteriorFactor, SchurSignConvention, SchurSourceBlock, SourceBlockSide,
    SpatialCutoffOperator, TransportedShiftOperator, WeightedDilationOperator,
    WienerHopfOperator, article_factor_membership_evidence,
    assemble_first_schur_correction, audit_common_algebra_membership,
    audit_relative_dilation_pair, audit_schur_cutoffs, certify_first_schur_pivot,
    derive_schur_correction_symbol,
)

gamma1 = gamma
gamma2 = sp.Symbol('gamma_2', positive=True)
U1 = WeightedDilationOperator(
    atom=OperatorAtom('U1'), gamma=gamma1, domain=space, codomain=space,
    assumptions=assumptions, source='article normalized branch-one dilation',
    evidence=('section 05 normalized dilation',),
)
U1_inverse_dilation = U1.inverse(OperatorAtom('U1_inverse_dilation'))
U2 = WeightedDilationOperator(
    atom=OperatorAtom('U2'), gamma=gamma2, domain=space, codomain=space,
    assumptions=assumptions, source='article normalized branch-two dilation',
    evidence=('section 05 normalized dilation',),
)

In [3]:
def typed_coefficient(name, scalar, role):
    return CoefficientMultiplicationOperator(
        multiplier(name, scalar, f'article coefficient {name}'), role,
    )

def exterior(index, branch_from, branch_to, dilation, polarity):
    rho = typed_coefficient(
        f'M_rho{index}', (dilation.gamma*x + 1)/(x + 1), f'transport rho_{index}',
    )
    shift = TransportedShiftOperator(
        atom=OperatorAtom(f'Vtilde_alpha{index}'), coefficient=rho, dilation=dilation,
        source=f'article transported shift {index}', evidence=('Vtilde=M_rho U',),
    )
    coefficient = typed_coefficient(f'G{index}', sp.Integer(index), f'G_{index}')
    difference = BranchDifferenceOperator(
        atom=OperatorAtom(f'D{index}'), transported_shift=shift, coefficient=coefficient,
        source=f'Vtilde_alpha{index}-G{index}', assumptions=assumptions,
        evidence=('article normalized exterior',),
    )
    wh = WienerHopfOperator(
        atom=OperatorAtom(f'W_core_{polarity}'), symbol=sp.exp(-sp.Abs(lam))+index*lam,
        frequency_variable=lam, domain=space, codomain=space,
        symbol_class='article exponential half-line symbol', source=f'article b_{polarity}',
        frequency_scaling_stable=True,
        stability_evidence=('half-line indicator scale invariance',),
        localization_controlled=True, evidence=('source symbol',),
    )
    chi = sp.Function('chi_infty')
    def cutoff(side):
        return SpatialCutoffOperator(
            atom=OperatorAtom(f'M_chi_{polarity}_{side}'), cutoff_symbol=chi(x),
            radial_variable=x, domain=space, codomain=space,
            support_region='transition on [1,2]', equals_one_near='[2,infinity)',
            equals_zero_away_from='[0,1]', coordinate_system='half-line radial',
            radial_scale=sp.Integer(1), source='article section 03:747-764',
            evidence=('chi_infty definition',),
        )
    localized = LocalizedOperator(
        atom=OperatorAtom(f'W{21 if index == 2 else 12}_{polarity}'),
        left_cutoff=cutoff('left'), core=wh, right_cutoff=cutoff('right'),
        source='article doubly localized Wiener--Hopf block',
        evidence=('section 06:60-95',),
    )
    return SchurExteriorFactor(
        atom=OperatorAtom('L21' if index == 2 else 'R12'), difference=difference,
        localized_wiener_hopf=localized, branch_from=Branch(branch_from),
        branch_to=Branch(branch_to), source='article exterior definition',
        assumptions=assumptions, evidence=('section 06 normalized block',),
    )

L21 = exterior(2, 1, 2, U2, 'minus')
R12 = exterior(1, 2, 1, U1, 'plus')
print('L21:', L21.source_expression, L21.branch_from, '->', L21.branch_to)
print('R12:', R12.source_expression, R12.branch_from, '->', R12.branch_to)

L21: Product(factors=(OperatorAtom(name='D2', kind='operator'), OperatorAtom(name='W21_minus', kind='operator'))) Branch(index=1) -> Branch(index=2)
R12: Product(factors=(OperatorAtom(name='D1', kind='operator'), OperatorAtom(name='W12_plus', kind='operator'))) Branch(index=2) -> Branch(index=1)


In [4]:
B21_minus = SchurSourceBlock(
    atom=OperatorAtom('B21_minus'), side=SourceBlockSide.LEFT, exterior=L21,
    interface=result.candidate_regularizer.left_interface,
    source='article section 06:382-390', evidence=('B21_minus=L21 T1_minus',),
)
B12_plus = SchurSourceBlock(
    atom=OperatorAtom('B12_plus'), side=SourceBlockSide.RIGHT, exterior=R12,
    interface=result.candidate_regularizer.right_interface_inverse,
    source='article section 06:391-399', evidence=('B12_plus=Z1_inverse R12',),
)
assembly = assemble_first_schur_correction(
    left_factor=L21, diagonal_regularizer=result, right_factor=R12,
    source_blocks=(B21_minus, B12_plus), assumptions=assumptions,
)
print('assembly:', assembly.assembly_status.value)
print('grouped:', assembly.grouped_expression)
print('source:', assembly.source_expression)
print('expanded:', assembly.expanded_expression)

assembly: CERTIFIED_EXACT_SCHUR_ASSEMBLY
grouped: Product(factors=(OperatorAtom(name='L21', kind='operator'), OperatorAtom(name='A11_factored_regularizer', kind='factored_regularizer'), OperatorAtom(name='R12', kind='operator')))
source: Product(factors=(OperatorAtom(name='B21_minus', kind='operator'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='B12_plus', kind='operator')))
expanded: Product(factors=(OperatorAtom(name='D2', kind='operator'), OperatorAtom(name='W21_minus', kind='operator'), OperatorAtom(name='T1_minus', kind='operator'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='Z1_inverse', kind='operator'), OperatorAtom(name='D1', kind='operator'), OperatorAtom(name='W12_plus', kind='operator')))


In [5]:
cutoff_records = audit_schur_cutoffs(assembly)
print('localizers:')
for record in cutoff_records:
    print(' ', record.branch.index, record.side.value, record.scale, record.status.value)

intervening = (
    OperatorAtom('P_plus'), result.candidate_regularizer.mellin_regularizer.atom,
    result.candidate_regularizer.right_interface_inverse.atom, R12.difference.atom,
)
dilation_audit = audit_relative_dilation_pair(
    U1_inverse_dilation, U1, intervening_factors=intervening,
    source='article seven-factor order', evidence=('no reordering theorem',),
)
print('relative dilation:', dilation_audit.status.value)
print('intervening:', [factor.name for factor in dilation_audit.intervening_factors])

localizers:
  2 LEFT 1 NO_REPLACEMENT_NEEDED
  1 RIGHT 1 NO_REPLACEMENT_NEEDED
  1 LEFT 1 NO_REPLACEMENT_NEEDED
  2 RIGHT 1 NO_REPLACEMENT_NEEDED
relative dilation: BLOCKED_BY_INTERVENING_FACTORS
intervening: ['P_plus', 'R1', 'Z1_inverse', 'D1']


In [6]:
factor_matrix = article_factor_membership_evidence(assembly)
for row in factor_matrix:
    print(row.factor_label, '|', row.classification.value, '|', row.candidate_algebra)
algebra = audit_common_algebra_membership(
    assembly=assembly, factor_evidence=factor_matrix, assumptions=assumptions,
    compact_ideal=compact_ideal, cutoff_audits=cutoff_records,
    dilation_audits=(dilation_audit,), evidence=('read-only literature audit',),
)
symbol_result = derive_schur_correction_symbol(
    assembly=assembly, algebra_certificate=algebra,
    cutoff_audits=cutoff_records, dilation_audits=(dilation_audit,),
)
print('common algebra:', algebra.status.value)
print('symbol:', symbol_result.symbol_status.value)
print('candidate symbol:', symbol_result.symbol)

Vtilde_alpha2-G2 | SOURCE_GAP | None
W21_minus | EXACT_THEOREM | Karlovich-2025 half-line algebra
T1_minus | SOURCE_GAP | None
R1 | MATCH_AFTER_NOTATION_TRANSLATION | KKL-2014 radial Mellin PDO algebra
Z1_inverse | DIRECT_ALGEBRAIC_CONSEQUENCE | Karlovich-2025 half-line algebra
Vtilde_alpha1-G1 | SOURCE_GAP | None
W12_plus | EXACT_THEOREM | Karlovich-2025 half-line algebra
common algebra: BLOCKED_BY_COMMON_ALGEBRA_SOURCE_GAP
symbol: BLOCKED_BY_ALGEBRA_MEMBERSHIP
candidate symbol: None


In [7]:
A22 = BranchedDiagonalOperator(
    atom=OperatorAtom('A22'), branch=Branch(2), domain=space, codomain=space,
    source='article section 06:27-42', evidence=('A22 exact definition',),
)
A21_model = OffDiagonalBlockModel(
    block_atom=OperatorAtom('A21'), branch_from=Branch(1), branch_to=Branch(2),
    domain=space, codomain=space, normalized_exterior=L21, model_sign=-1,
    compact_ideal=compact_ideal, source='article section 06:86-95',
    evidence=('A21 is minus L21 modulo compact',),
)
A12_model = OffDiagonalBlockModel(
    block_atom=OperatorAtom('A12'), branch_from=Branch(2), branch_to=Branch(1),
    domain=space, codomain=space, normalized_exterior=R12, model_sign=1,
    compact_ideal=compact_ideal, source='article section 06:78-85',
    evidence=('A12 is plus R12 modulo compact',),
)
sign = SchurSignConvention(
    raw_schur_sign=-1, left_model_sign=-1, right_model_sign=1,
    correction_sign_in_model_pivot=1, source='article sections 06:281-354',
    evidence=('external Schur minus plus normalized A21 minus',),
)
pivot = certify_first_schur_pivot(
    diagonal_block=A22, correction=assembly, left_block=A21_model,
    right_block=A12_model, sign=sign, symbol_result=symbol_result,
)
print('raw pivot:', pivot.source_expression)
print('model pivot:', pivot.pivot_expression)
print('relation:', type(pivot.pivot_relation).__name__)
print('Fredholm:', pivot.fredholm_status.value)
print('LaTeX:', pivot.latex)

raw pivot: LinearCombination(terms=(Term(coefficient=1, product=Product(factors=(OperatorAtom(name='A22', kind='operator'),))), Term(coefficient=-1, product=Product(factors=(OperatorAtom(name='A21', kind='operator'), OperatorAtom(name='A11_factored_regularizer', kind='factored_regularizer'), OperatorAtom(name='A12', kind='operator'))))))
model pivot: LinearCombination(terms=(Term(coefficient=1, product=Product(factors=(OperatorAtom(name='A22', kind='operator'),))), Term(coefficient=1, product=Product(factors=(OperatorAtom(name='C22_1', kind='schur_correction'),)))))
relation: ModCompactEquivalence
Fredholm: NOT_CERTIFIED
LaTeX: \mathcal A_{2,2}^{(1)}=\mathcal A_{2,2}-\mathcal A_{2,1}\mathcal A_{1,1}^{(-1)}\mathcal A_{1,2}\simeq\mathcal A_{2,2}+\mathcal C_{2,2}^{(1)}


In [8]:
print('open obligations:')
for obligation in assembly.proof_obligations:
    print(' -', obligation.key)
print('assembly LaTeX:', assembly.latex)
print('pivot LaTeX:', pivot.latex)

open obligations:
 - full_regularizer_algebra_membership
 - cutoff_replacement_mod_compacts
 - off_diagonal_wh_membership
 - relative_dilation_membership
 - wh_mellin_wh_sandwich_membership
 - fredholm_algebra_membership
 - schur_correction_symbol
 - first_schur_pivot_fredholmness
assembly LaTeX: \mathcal C_{2,2}^{(1)}=\mathcal L_{2,1}(\mathcal T_{1,-}\mathcal R_1Z_1^{-1})\mathcal R_{1,2}\quad\text{CERTIFIED_EXACT_SCHUR_ASSEMBLY}
pivot LaTeX: \mathcal A_{2,2}^{(1)}=\mathcal A_{2,2}-\mathcal A_{2,1}\mathcal A_{1,1}^{(-1)}\mathcal A_{1,2}\simeq\mathcal A_{2,2}+\mathcal C_{2,2}^{(1)}
